In [0]:
from pyspark.sql.functions import col, current_timestamp
RAW = "/Volumes/workspace/public_health/raw"

who = (
    spark.read
    .option("multiLine", "true")
    .json(f"{RAW}/who/*.json")
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

who.printSchema()
print("rows:", who.count())
display(who.limit(5))

In [0]:
(
    who.write
    .mode("overwrite")
    .saveAsTable("workspace.public_health.bronze_who")
)
print("written")

In [0]:
%sql
SELECT IndicatorCode, COUNT(*) AS rows, MIN(TimeDim) AS first_year, MAX(TimeDim) AS last_year
FROM workspace.public_health.bronze_who
GROUP BY IndicatorCode
ORDER BY IndicatorCode;

In [0]:
%sql
DESCRIBE HISTORY workspace.public_health.bronze_who;

In [0]:
wb = (
    spark.read
    .option("multiLine", "true")
    .json(f"{RAW}/worldbank/*.json")
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

wb.printSchema()
print("rows:", wb.count())

wb.write.mode("overwrite").saveAsTable("workspace.public_health.bronze_worldbank")
print("written")

In [0]:
%sql
SELECT indicator.id AS indicator, COUNT(*) AS rows,
       COUNT(value) AS non_null_values,
       MIN(date) AS first_year, MAX(date) AS last_year
FROM workspace.public_health.bronze_worldbank
GROUP BY indicator.id
ORDER BY indicator.id;

In [0]:
OWID_CHARTS = [
    "life-expectancy",
    "child-mortality",
    "share-of-children-vaccinated-against-measles",
]

for slug in OWID_CHARTS:
    table_name = "bronze_owid_" + slug.replace("-", "_")
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW}/owid/{slug}_*.csv")
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingested_at", current_timestamp())
    )
    df.write.mode("overwrite").saveAsTable(f"workspace.public_health.{table_name}")
    print(f"{table_name}: {df.count()} rows, columns {df.columns}")

In [0]:
%sql
SHOW TABLES IN workspace.public_health;